In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

# -----------------------------
# Paths
# -----------------------------
base_dir = Path.home() / "Downloads" / "fide_history"

sample1_path = base_dir / "player_sample_1000.csv"
sample2_path = base_dir / "player_sample_1000_set2.csv"
sample3_path = base_dir / "player_sample_1000_set3.csv"

# -----------------------------
# Load first sample
# -----------------------------
sample1 = pd.read_csv(sample1_path)

sample1.columns = sample1.columns.str.strip()
sample1["ID Number"] = sample1["ID Number"].astype(str).str.strip()

sample1_ids = set(sample1["ID Number"])

print("Sample 1 players:", len(sample1_ids))

Sample 1 players: 1000


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

base_dir = Path.home() / "Downloads" / "fide_history"

for f in base_dir.glob("*.csv"):
    print(f.name)

player_sample_1000.csv
player_sample_1000_set3.csv
player_sample_1000_set2.csv
player_sample_remaining_7419_2.csv
chess_results_clean.csv
player_sample_remaining_7419_1.csv
player_sample_1000_1.csv
player_sample_1000_2.csv
chess_results_india_players.csv
fide_calculations_all_players_standard_clean_combined.csv
player_features_full_pool.csv
fide_standard_history_clean.csv
player_sample_remaining_7419.csv


In [8]:
history_path = base_dir / "fide_standard_history_clean.csv"

history_df = pd.read_csv(history_path)

history_df.columns = history_df.columns.str.strip()

print(history_df.shape)
print(history_df.columns.tolist())
display(history_df.head())

(13858699, 11)
['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'standard_rating', 'Gms', 'K', 'B-day', 'rating_month', 'source_file']


,ID Number,Name,Fed,Sex,Tit,standard_rating,Gms,K,B-day,rating_month,source_file
0,10245154,"A B M Jobair, Hossain",BAN,M,NaN,1583,0,40,1998,2024-01,standard_2024_01.zip
1,25121731,A C J John,IND,M,NaN,1063,0,40,1987,2024-01,standard_2024_01.zip
2,35077023,A Chakravarthy,IND,M,NaN,1151,0,40,1986,2024-01,standard_2024_01.zip
3,10207538,"A E M, Doshtagir",BAN,M,NaN,1840,0,40,1974,2024-01,standard_2024_01.zip
4,10680810,"A hamed Ashraf, Abdallah",EGY,M,NaN,1728,0,40,2001,2024-01,standard_2024_01.zip


In [10]:
history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], errors="coerce")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce").fillna(0)
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

history_df["Birth_Year"] = history_df["B-day"].replace(0, np.nan)

In [12]:
# Total unique players in full cleaned FIDE Standard rating history
total_unique_players = history_df["ID Number"].astype(str).str.strip().nunique()

print("Total unique FIDE Standard players in cleaned history:", total_unique_players)

Total unique FIDE Standard players in cleaned history: 573377


In [14]:
# Total unique Indian FIDE Standard players, all ages
total_indian_players = (
    history_df[history_df["Fed"] == "IND"]["ID Number"]
    .astype(str)
    .str.strip()
    .nunique()
)

print("Total Indian FIDE Standard players, all ages:", total_indian_players)

Total Indian FIDE Standard players, all ages: 56950


In [20]:
india_youth_history = history_df[
    (history_df["Fed"] == "IND") &
    (history_df["Birth_Year"].notna()) &
    (history_df["Birth_Year"] >= 2008)
].copy()

print("Indian youth player-month records:", india_youth_history.shape)
print("Unique Indian youth players:", india_youth_history["ID Number"].nunique())

Indian youth player-month records: (305876, 12)
Unique Indian youth players: 18771


In [22]:
feature_start = pd.to_datetime("2024-04-01")
cutoff_month = pd.to_datetime("2025-04-01")
outcome_end = pd.to_datetime("2026-04-01")

study_df = india_youth_history[
    (india_youth_history["rating_month"] >= feature_start) &
    (india_youth_history["rating_month"] <= outcome_end)
].copy()

print(study_df["rating_month"].min(), study_df["rating_month"].max())
print("Unique players:", study_df["ID Number"].nunique())

2024-04-01 00:00:00 2026-04-01 00:00:00
Unique players: 18502


In [24]:
def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) > 0 else np.nan

def last_valid(series):
    valid = series.dropna()
    return valid.iloc[-1] if len(valid) > 0 else np.nan

def rating_at_month(df, month):
    temp = df[df["rating_month"] == month][["ID Number", "standard_rating"]].copy()
    temp = temp.rename(columns={"standard_rating": f"rating_{month.strftime('%Y_%m')}"})
    return temp

In [26]:
rating_apr2024 = rating_at_month(study_df, feature_start)
rating_apr2025 = rating_at_month(study_df, cutoff_month)
rating_apr2026 = rating_at_month(study_df, outcome_end)

In [28]:
feature_window_df = study_df[
    (study_df["rating_month"] >= feature_start) &
    (study_df["rating_month"] <= cutoff_month)
].copy()

player_features = (
    feature_window_df
    .sort_values(["ID Number", "rating_month"])
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("Birth_Year", last_valid),
        k_factor_cutoff=("K", "last"),
        games_12m=("Gms", "sum"),
        avg_monthly_games_12m=("Gms", "mean"),
        max_monthly_games_12m=("Gms", "max"),
        active_months_12m=("Gms", lambda x: (x > 0).sum()),
        inactive_months_12m=("Gms", lambda x: (x == 0).sum()),
        months_available=("rating_month", "nunique")
    )
)

In [30]:
player_features = (
    player_features
    .merge(rating_apr2024, on="ID Number", how="left")
    .merge(rating_apr2025, on="ID Number", how="left")
    .merge(rating_apr2026, on="ID Number", how="left")
)

player_features = player_features.rename(columns={
    "rating_2024_04": "rating_apr2024",
    "rating_2025_04": "rating_start",
    "rating_2026_04": "rating_end"
})

player_features["rating_change_12m"] = (
    player_features["rating_start"] - player_features["rating_apr2024"]
)

player_features["rating_gain_12m"] = (
    player_features["rating_end"] - player_features["rating_start"]
)

player_features["target_gain_12m"] = player_features["rating_gain_12m"]

player_features["age_at_cutoff"] = 2025 - player_features["Birth_Year"]

player_features["cross_1800_12m"] = (
    (player_features["rating_start"] < 1800) &
    (player_features["rating_end"] >= 1800)
).astype(int)

player_features["cross_2000_12m"] = (
    (player_features["rating_start"] < 2000) &
    (player_features["rating_end"] >= 2000)
).astype(int)

In [32]:
def get_rating_change(df, start_month, end_month, col_name):
    start = rating_at_month(df, start_month).rename(
        columns={f"rating_{start_month.strftime('%Y_%m')}": "rating_start_temp"}
    )
    end = rating_at_month(df, end_month).rename(
        columns={f"rating_{end_month.strftime('%Y_%m')}": "rating_end_temp"}
    )

    temp = start.merge(end, on="ID Number", how="outer")
    temp[col_name] = temp["rating_end_temp"] - temp["rating_start_temp"]

    return temp[["ID Number", col_name]]

rating_change_3m = get_rating_change(
    study_df,
    pd.to_datetime("2025-01-01"),
    pd.to_datetime("2025-04-01"),
    "rating_change_3m"
)

rating_change_6m = get_rating_change(
    study_df,
    pd.to_datetime("2024-10-01"),
    pd.to_datetime("2025-04-01"),
    "rating_change_6m"
)

player_features = (
    player_features
    .merge(rating_change_3m, on="ID Number", how="left")
    .merge(rating_change_6m, on="ID Number", how="left")
)

In [34]:
def games_in_window(df, start_month, end_month, col_name):
    temp = df[
        (df["rating_month"] >= start_month) &
        (df["rating_month"] <= end_month)
    ].copy()

    out = (
        temp.groupby("ID Number", as_index=False)
        .agg(**{col_name: ("Gms", "sum")})
    )

    return out

games_3m = games_in_window(
    study_df,
    pd.to_datetime("2025-01-01"),
    pd.to_datetime("2025-04-01"),
    "games_3m"
)

games_6m = games_in_window(
    study_df,
    pd.to_datetime("2024-10-01"),
    pd.to_datetime("2025-04-01"),
    "games_6m"
)

player_features = (
    player_features
    .merge(games_3m, on="ID Number", how="left")
    .merge(games_6m, on="ID Number", how="left")
)

player_features[["games_3m", "games_6m"]] = player_features[["games_3m", "games_6m"]].fillna(0)

In [36]:
temp = feature_window_df.sort_values(["ID Number", "rating_month"]).copy()
temp["monthly_rating_change"] = temp.groupby("ID Number")["standard_rating"].diff()

rating_volatility = (
    temp.groupby("ID Number", as_index=False)
    .agg(rating_volatility_12m=("monthly_rating_change", "std"))
)

player_features = player_features.merge(rating_volatility, on="ID Number", how="left")

In [38]:
player_features["rating_band"] = pd.cut(
    player_features["rating_start"],
    bins=[1399, 1600, 1800, 2000, 3000],
    labels=["1400-1599", "1600-1799", "1800-1999", "2000+"],
    include_lowest=True
)

print(player_features["rating_band"].value_counts(dropna=False).sort_index())

rating_band
1400-1599    8879
1600-1799    1427
1800-1999     354
2000+         181
NaN           911
Name: count, dtype: int64


In [40]:
player_features = player_features[
    player_features["rating_start"].notna() &
    player_features["rating_end"].notna()
].copy()

player_features = player_features.drop_duplicates(subset=["ID Number"]).copy()

print("Final full player_features shape:", player_features.shape)
print("Unique players:", player_features["ID Number"].nunique())
display(player_features.head())

Final full player_features shape: (10419, 28)
Unique players: 10419


,ID Number,Name,Fed,Sex,Title,Birth_Year,k_factor_cutoff,games_12m,avg_monthly_games_12m,max_monthly_games_12m,active_months_12m,inactive_months_12m,months_available,rating_apr2024,rating_start,rating_end,rating_change_12m,rating_gain_12m,target_gain_12m,age_at_cutoff,cross_1800_12m,cross_2000_12m,rating_change_3m,rating_change_6m,games_3m,games_6m,rating_volatility_12m,rating_band
0,129105193,"Tewari, Siddhant",IND,M,None,2012.0,40,44,3.384615,13,6,7,13,1479.0,1561.0,1535.0,82.0,-26.0,-26.0,13.0,0,0,-56.0,119.0,15.0,36.0,88.346254,1400-1599
1,13214624,"Daivik, Dayanand",IND,M,None,2013.0,40,9,0.692308,5,3,10,13,1454.0,1466.0,1476.0,12.0,10.0,10.0,12.0,0,0,0.0,19.0,5.0,5.0,23.013830,1400-1599
2,14332612,"Sopanam Prabha, Arjun",IND,M,None,2008.0,40,0,0.000000,0,0,13,13,1425.0,1425.0,1425.0,0.0,0.0,0.0,17.0,0,0,0.0,0.0,0.0,0.0,0.000000,1400-1599
3,14350670,"Divakaran, Sinddhizhai",IND,F,None,2013.0,40,50,4.166667,12,9,3,12,1425.0,1501.0,1469.0,76.0,-32.0,-32.0,12.0,0,0,61.0,9.0,25.0,38.0,69.490222,1400-1599
4,146105007,"Deshmukh, Omm",IND,M,None,2012.0,40,50,3.846154,12,7,6,13,1441.0,1557.0,1654.0,116.0,97.0,97.0,13.0,0,0,-14.0,155.0,11.0,26.0,60.204701,1400-1599


In [42]:
player_features_path = base_dir / "player_features_full_pool.csv"

player_features.to_csv(player_features_path, index=False)

print("Saved full player feature pool:", player_features_path)

Saved full player feature pool: /Users/arunkumar/Downloads/fide_history/player_features_full_pool.csv


In [44]:
sample1_path = base_dir / "player_sample_1000.csv"

sample1 = pd.read_csv(sample1_path)
sample1.columns = sample1.columns.str.strip()
sample1["ID Number"] = sample1["ID Number"].astype(str).str.strip()

sample1_ids = set(sample1["ID Number"])

print("Sample 1 players:", len(sample1_ids))
print("Full pool players:", player_features["ID Number"].nunique())

Sample 1 players: 1000
Full pool players: 10419


In [46]:
def create_stratified_sample(
    source_df,
    exclude_ids=None,
    sample_size=1000,
    stratify_col="rating_band",
    random_state=42
):
    if exclude_ids is None:
        exclude_ids = set()
    else:
        exclude_ids = set(str(x).strip() for x in exclude_ids)

    df = source_df.copy()
    df["ID Number"] = df["ID Number"].astype(str).str.strip()
    df = df.drop_duplicates(subset=["ID Number"]).copy()

    df = df[~df["ID Number"].isin(exclude_ids)].copy()

    if len(df) < sample_size:
        raise ValueError(f"Only {len(df)} players available after exclusion.")

    band_counts = df[stratify_col].value_counts(dropna=False)
    band_props = band_counts / band_counts.sum()
    target_counts = (band_props * sample_size).round().astype(int)

    diff = sample_size - target_counts.sum()

    if diff != 0:
        sorted_bands = target_counts.sort_values(ascending=False).index.tolist()
        for i in range(abs(diff)):
            band = sorted_bands[i % len(sorted_bands)]
            if diff > 0:
                target_counts.loc[band] += 1
            elif target_counts.loc[band] > 0:
                target_counts.loc[band] -= 1

    sample_parts = []

    for band, n in target_counts.items():
        band_df = df[df[stratify_col] == band].copy()
        if n > 0 and not band_df.empty:
            sample_parts.append(
                band_df.sample(n=min(n, len(band_df)), random_state=random_state)
            )

    sample = pd.concat(sample_parts, ignore_index=True)

    if len(sample) < sample_size:
        needed = sample_size - len(sample)
        remaining = df[~df["ID Number"].isin(sample["ID Number"])]
        extra = remaining.sample(n=needed, random_state=random_state + 999)
        sample = pd.concat([sample, extra], ignore_index=True)

    if len(sample) > sample_size:
        sample = sample.sample(n=sample_size, random_state=random_state)

    return sample.reset_index(drop=True)

In [48]:
sample2 = create_stratified_sample(
    source_df=player_features,
    exclude_ids=sample1_ids,
    sample_size=1000,
    stratify_col="rating_band",
    random_state=202
)

sample2_ids = set(sample2["ID Number"])

sample3 = create_stratified_sample(
    source_df=player_features,
    exclude_ids=sample1_ids.union(sample2_ids),
    sample_size=1000,
    stratify_col="rating_band",
    random_state=303
)

sample3_ids = set(sample3["ID Number"])

print("Sample 1:", len(sample1_ids))
print("Sample 2:", len(sample2_ids))
print("Sample 3:", len(sample3_ids))

print("Overlap sample1-sample2:", len(sample1_ids & sample2_ids))
print("Overlap sample1-sample3:", len(sample1_ids & sample3_ids))
print("Overlap sample2-sample3:", len(sample2_ids & sample3_ids))

Sample 1: 1000
Sample 2: 1000
Sample 3: 1000
Overlap sample1-sample2: 0
Overlap sample1-sample3: 0
Overlap sample2-sample3: 0


In [50]:
sample2_path = base_dir / "player_sample_1000_set2.csv"
sample3_path = base_dir / "player_sample_1000_set3.csv"

#sample2.to_csv(sample2_path, index=False)
#sample3.to_csv(sample3_path, index=False)

#print("Saved:", sample2_path)
#print("Saved:", sample3_path)

Saved: /Users/arunkumar/Downloads/fide_history/player_sample_1000_set2.csv
Saved: /Users/arunkumar/Downloads/fide_history/player_sample_1000_set3.csv


In [52]:
report_rows = []

def add_numeric_row(var, label):
    if var in df.columns:
        s = pd.to_numeric(df[var], errors="coerce")
        report_rows.append({
            "Variable Group": label,
            "Variable": var,
            "N": s.notna().sum(),
            "Missing %": round(s.isna().mean() * 100, 2),
            "Mean": round(s.mean(), 2),
            "Median": round(s.median(), 2),
            "Min": round(s.min(), 2),
            "Max": round(s.max(), 2)
        })

add_numeric_row("age_at_cutoff", "Player profile")
add_numeric_row("rating_start", "Baseline rating")
add_numeric_row("k_factor_cutoff", "Baseline rating")
add_numeric_row("total_games_12m", "Activity")
add_numeric_row("active_months_12m", "Activity")
add_numeric_row("inactive_months_12m", "Activity")
add_numeric_row("rating_change_3m", "Rating momentum")
add_numeric_row("rating_change_6m", "Rating momentum")
add_numeric_row("rating_change_12m", "Rating momentum")
add_numeric_row("calc_games_12m", "Enrichment")
add_numeric_row("calc_tournaments_12m", "Enrichment")
add_numeric_row("avg_opponent_rating_12m", "Opponent strength")
add_numeric_row("score_percentage_12m", "Performance quality")
add_numeric_row("rating_gain_12m", "Outcome")

report_univariate_numeric = pd.DataFrame(report_rows)

display(report_univariate_numeric)

NameError: name 'df' is not defined

##### create rest of the data excluding these 3 set of players ie 3000 players to download separately

In [2]:
# ============================================================
# Create remaining player list after excluding Sample 1, 2, and 3
# ============================================================

from pathlib import Path
import pandas as pd

base_dir = Path.home() / "Downloads" / "fide_history"

# Load full engineered pool
full_pool_path = base_dir / "player_features_full_pool.csv"
player_features = pd.read_csv(full_pool_path)

player_features.columns = player_features.columns.str.strip()
player_features["ID Number"] = player_features["ID Number"].astype(str).str.strip()

# Load already created/downloaded samples
sample1 = pd.read_csv(base_dir / "player_sample_1000.csv")
sample2 = pd.read_csv(base_dir / "player_sample_1000_set2.csv")
sample3 = pd.read_csv(base_dir / "player_sample_1000_set3.csv")

for df in [sample1, sample2, sample3]:
    df.columns = df.columns.str.strip()
    df["ID Number"] = df["ID Number"].astype(str).str.strip()

# Combine already downloaded player IDs
downloaded_ids = (
    set(sample1["ID Number"]) |
    set(sample2["ID Number"]) |
    set(sample3["ID Number"])
)

print("Full pool players:", player_features["ID Number"].nunique())
print("Already downloaded/sample players:", len(downloaded_ids))

# Exclude already downloaded 3,000 players
remaining_players = player_features[
    ~player_features["ID Number"].isin(downloaded_ids)
].copy()

remaining_players = remaining_players.drop_duplicates(subset=["ID Number"]).copy()

print("Remaining players:", remaining_players["ID Number"].nunique())

# Save remaining player list
remaining_path = base_dir / "player_sample_remaining_7419.csv"
remaining_players.to_csv(remaining_path, index=False)

print("Saved remaining player file to:")
print(remaining_path)

display(remaining_players.head())


Full pool players: 10419
Already downloaded/sample players: 3000
Remaining players: 7845
Saved remaining player file to:
/Users/arunkumar/Downloads/fide_history/player_sample_remaining_7419.csv


,ID Number,Name,Fed,Sex,Title,Birth_Year,k_factor_cutoff,games_12m,avg_monthly_games_12m,max_monthly_games_12m,...,target_gain_12m,age_at_cutoff,cross_1800_12m,cross_2000_12m,rating_change_3m,rating_change_6m,games_3m,games_6m,rating_volatility_12m,rating_band
0,129105193,"Tewari, Siddhant",IND,M,NaN,2012.0,40,44,3.384615,13,...,-26.0,13.0,0,0,-56.0,119.0,15.0,36.0,88.346254,1400-1599
1,13214624,"Daivik, Dayanand",IND,M,NaN,2013.0,40,9,0.692308,5,...,10.0,12.0,0,0,0.0,19.0,5.0,5.0,23.013830,1400-1599
2,14332612,"Sopanam Prabha, Arjun",IND,M,NaN,2008.0,40,0,0.000000,0,...,0.0,17.0,0,0,0.0,0.0,0.0,0.0,0.000000,1400-1599
4,146105007,"Deshmukh, Omm",IND,M,NaN,2012.0,40,50,3.846154,12,...,97.0,13.0,0,0,-14.0,155.0,11.0,26.0,60.204701,1400-1599
5,146105016,"Ramesh Nair, Advait",IND,M,NaN,2011.0,40,37,2.846154,10,...,283.0,14.0,1,0,-88.0,34.0,14.0,24.0,39.637754,1600-1799


In [4]:
remaining_players.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7845 entries, 0 to 10418
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID Number              7845 non-null   object 
 1   Name                   7845 non-null   object 
 2   Fed                    7845 non-null   object 
 3   Sex                    7845 non-null   object 
 4   Title                  76 non-null     object 
 5   Birth_Year             7845 non-null   float64
 6   k_factor_cutoff        7845 non-null   int64  
 7   games_12m              7845 non-null   int64  
 8   avg_monthly_games_12m  7845 non-null   float64
 9   max_monthly_games_12m  7845 non-null   int64  
 10  active_months_12m      7845 non-null   int64  
 11  inactive_months_12m    7845 non-null   int64  
 12  months_available       7845 non-null   int64  
 13  rating_apr2024         4953 non-null   float64
 14  rating_start           7845 non-null   float64
 15  rating_e